In [ ]:
import pandas
from matplotlib import pyplot
import numpy
def confusion_matrix(profile, ax0=None, yticklabels=["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]):
    data = pandas.read_csv("confusion.csv")
    data["confusion"] = data["confusion"].apply(eval)
    cm0 = data[(data["id"] == profile) & (data["seed"] == 0)].iloc[0]["confusion"]
    cm1 = data[(data["id"] == profile) & (data["seed"] == 1)].iloc[0]["confusion"]
    cm2 = data[(data["id"] == profile) & (data["seed"] == 2)].iloc[0]["confusion"]
    cm3 = data[(data["id"] == profile) & (data["seed"] == 3)].iloc[0]["confusion"]
    cm4 = data[(data["id"] == profile) & (data["seed"] == 4)].iloc[0]["confusion"]
    cm5 = data[(data["id"] == profile) & (data["seed"] == 5)].iloc[0]["confusion"]
    cm = numpy.rint(numpy.mean([cm0, cm1, cm2, cm3, cm4, cm5], axis=0)).astype(int)
    ax = pyplot.subplots(figsize=(5, 4))[1] if ax0 is None else ax0
    image = ax.imshow(cm, cmap="Blues")
    for row in range(len(cm)):
        for col in range(len(cm[row])):
            ax.text(col, row, cm[row][col], ha="center", va="center",
                color="white" if cm[row][col] > max(max(row) for row in cm) / 2 else "black")
    ax.set_title(profile + " profile")
    ax.set_xticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
    ax.set_yticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
    ax.set_xticklabels(["a", "a", "b", "c", "d", "d", "f", "h", "s", "t"])
    ax.set_yticklabels(yticklabels)
    if ax0 is None:
        pyplot.tight_layout()
        pyplot.show()

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(11, 4))
confusion_matrix("fixed", ax1)
confusion_matrix("dpa-dp-0.85", ax2)

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(11, 4))
confusion_matrix("dpa-dp-0.75", ax1)
confusion_matrix("dpa-dp-0.65", ax2)

In [ ]:
fig, (ax1, ax2, ax3) = pyplot.subplots(1, 3, figsize=(12, 4))
confusion_matrix("fixed", ax1)
confusion_matrix("dpa-dp-0.75", ax2, yticklabels=["a", "a", "b", "c", "d", "d", "f", "h", "s", "t"])
confusion_matrix("dpa-dp-0.65", ax3, yticklabels=["a", "a", "b", "c", "d", "d", "f", "h", "s", "t"])
# fig.savefig("cifar_confusion.pdf", bbox_inches="tight")

In [ ]:
import ast
import numpy as np
import pandas
from scipy import stats
# FIXME slop

ALL, DIS = np.arange(10), np.array([2, 4])

def macrof1():
    df = pandas.read_csv("confusion.csv")
    df["cm"] = df["confusion"].apply(lambda s: np.array(ast.literal_eval(s), dtype=float))
    
    def macro_f1(cm, classes):
        tp = np.diag(cm)[classes]
        col = cm[:, classes].sum(0)
        row = cm[classes, :].sum(1)
        prec = np.divide(tp, col, out=np.zeros(len(classes)), where=col > 0)
        rec = np.divide(tp, row, out=np.zeros(len(classes)), where=row > 0)
        s = prec + rec
        f1 = np.divide(2 * prec * rec, s, out=np.zeros(len(classes)), where=s > 0)
        return 100 * f1.mean()  # percentage points
    
    def half_width(x):  # 95% t-interval half-width
        return stats.t.ppf(0.975, len(x) - 1) * np.std(x, ddof=1) / np.sqrt(len(x))
    
    def fmt(x, signed=False):
        m, h = np.mean(x), half_width(x)
        return f"{m:+.2f} ± {h:.2f}" if signed else f"{m:.2f} ± {h:.2f}"
    
    def per_seed(profile, classes):
        d = df[df["id"] == profile].sort_values("seed")
        return np.array([macro_f1(c, classes) for c in d["cm"]]), d["seed"].values
    
    profiles = ["fixed"] + [p for p in df["id"].unique() if p != "fixed"]
    base_all, base_seeds = per_seed("fixed", ALL)
    base_dis, _ = per_seed("fixed", DIS)
    
    rows = []
    for p in profiles:
        a, seeds = per_seed(p, ALL)
        d, _ = per_seed(p, DIS)
        assert (seeds == base_seeds).all(), f"{p}: seed order differs from fixed"
        diff_a, diff_d = fmt(a - base_all, True), fmt(d - base_dis, True)
        rows.append({"profile": p, "overall": fmt(a), "diff": diff_a,
                     "disaster": fmt(d), "disaster diff": diff_d})
    return pandas.DataFrame(rows).set_index("profile")
print(macrof1().to_latex().replace("±", r"$\pm$"))